# Diagrama UML

books
- id (PK)
- title
- url
- price
- rating (1–5)
- stock (0–1)
- description
- category_id (FK → categories.id)

authors
- id (PK)
- name (UNIQUE)

book_author
- book_id (FK → books.id)
- author_id (FK → authors.id)

categories
- id (PK)
- name (UNIQUE)

![alt text](image-1.png)

In [ ]:
from bs4 import BeautifulSoup
import requests
from urllib.parse import urljoin
import json, os, time, re, sys



# Cambio por url de pagina 1 para despues ir cambiando a cada pagina
BASE = "https://books.toscrape.com/"
url = urljoin(BASE, "catalogue/page-1.html")
next_page = url



rating_number = {"One":1,"Two":2,"Three":3,"Four":4,"Five":5}
items = []


OPENLIB_SEARCH = "https://openlibrary.org/search.json"
GOOGLE_BOOKS = "https://www.googleapis.com/books/v1/volumes"

# funcion para obtener autores usando Open Library y Google Books
def get_authors(title, category, lang="en", timeout=1, max_authors=3):
    """
    Devuelve una LISTA de autores usando Open Library y como fallback Google Books.
    Siempre retorna al menos ["Desconocido"] si no encuentra nada.
    """
    def _norm(name: str) -> str:
        return re.sub(r"\s+", " ", name).strip()

    autores = []

    # 1) Open Library (puede traer múltiples)
    try:
        ol = requests.get(
            "https://openlibrary.org/search.json",
            params={"title": title, "limit": 3},
            timeout=timeout
        )
        ol.raise_for_status()
        docs = ol.json().get("docs", [])
        for d in docs:
            for a in d.get("author_name", []) or []:
                a = _norm(a)
                if a and a not in autores:
                    autores.append(a)
                    if len(autores) >= max_authors:
                        break
            if len(autores) >= max_authors:
                break
    except requests.RequestException as e:
        print(f"[⚠️] OpenLibrary error: {e}")


    return autores if autores else ["Desconocido"]


# funcion para obtener el soup de una pagina o una url
def get_soup(url_pagina_soup):
    for intento in range(5):  # 🔹 ADICIÓN: reintentos
        try:
            resp = requests.get(url_pagina_soup, timeout=20)
            resp.raise_for_status()
            return BeautifulSoup(resp.text, "lxml")
        except requests.RequestException as e:
            print(f"[WARN] Falló {url_pagina_soup} (intento {intento+1}): {e}")
            time.sleep(0.5)
    raise RuntimeError(f"No pude obtener {url_pagina_soup}")



# mientras existan paginas siguientes, veridicado por boton next. otra forma de hacerlo es iterando
while next_page:

    soup = get_soup(next_page)
    books = soup.find_all("article", class_="product_pod")

    for b in books:


        # Title
        title = b.h3.a["title"]


        # Price
        # 💥💥💥 
        txtprice = b.find("p", class_="price_color").get_text(strip=True)
        price = float(re.sub(r"[^\d.]", "", txtprice))


        # URL
        href = b.h3.a["href"]
        book_url = urljoin(next_page, href)

        try:
            # detalle 
            soup_detail = get_soup(book_url)
        except RuntimeError as e:
            print(f"[SKIP] No pude abrir detalle: {book_url} -> {e}")
            continue   

        # category
        breadcrumb_link = soup_detail.select("ul.breadcrumb li a")
        category = breadcrumb_link[-1].get_text(strip=True) if len(breadcrumb_link) >= 3 else "Unknown"


        # Rating                   
        rating_tag = soup_detail.select_one("p.star-rating")
        classes = rating_tag.get("class", []) if rating_tag else []                        
        rating_word = next((c for c in classes if c in rating_number), "One")
        rating = rating_number.get(rating_word, 1)                         


        # Stock
        stock_text = soup_detail.find("p", class_="instock availability").get_text(strip=True)
        stock_text = stock_text.lower().replace(" ", "")
        stock = int(1 if "instock" in stock_text else 0)






        # 🔹 ADICIÓN (llamar API): buscar autores por título
        # Explicación:
        # - Usamos la función get_autor(title, category).
        # - Retorna una lista [] con los nombres de los autores o Desconocido.
        # - Se agrega como campo "author" en el item para persistirlo luego en JSON.
        authors = get_authors(title, category)
        # Pequeña pausa de cortesía para no saturar la API si hay muchos libros
        time.sleep(0.15)

        items.append({
            "title": title,
            "category": category,
            "rating": rating,
            "URL": book_url,
            "price": price,
            "stock": stock,
            # 🔹 ADICIÓN (nuevo campo en el dataset): autores como lista
            # Explicación:
            # - Este campo nuevo te permitirá luego crear tablas 'autores' y 'libro_autor' (M:N) en tu DB.
            # - Mantenerlo como lista te conserva todos los coautores que reporte la API.
            "authors": authors
        })
        print(items)

    # para cada pagina del 1 al 5
    botton_next = soup.find("li", class_="next")
    if botton_next:
        href_next = botton_next.a["href"]
        next_page = urljoin(next_page, href_next)
        time.sleep(0.15)
    else:
        break


# Guardar en JSON
with open('libros_scrapeados.json', 'w', encoding='utf-8') as f:
    json.dump(items, f, ensure_ascii=False, indent=2)

# Leer desde JSON
with open('libros_scrapeados.json', 'r', encoding='utf-8') as f:
    datos = json.load(f)
    print(f'Se guardaron {len(datos)} libros')
    print(datos[0])  # Mostrar el primero para validar estructura

# Abrir automáticamente el archivo (solo en Windows)
os.startfile('libros_scrapeados.json')


[{'title': 'A Light in the Attic', 'category': 'Poetry', 'rating': 3, 'URL': 'https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html', 'price': 51.77, 'stock': 1, 'description': "It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love that Silverstein. Need proof of his genius? RockabyeR

RuntimeError: No pude obtener https://books.toscrape.com/catalogue/page-3.html

In [ ]:
import sqlite3

# 1. Conectamos con la base
# conn representa la conexión a la base de datos.
conn = sqlite3.connect("libros.db")
cursor = conn.cursor()


# 2. Activamos las llaves foráneas (para relaciones entre tablas)
conn.execute("PRAGMA foreign_keys = ON;")   # PRAGMA es una directiva especial de SQLite para configurar opciones




# 3. Escribimos el DDL (definición de tablas)
DDL = """
CREATE TABLE IF NOT EXISTS categories (
    id      INTEGER PRIMARY KEY,
    name    TEXT NOT NULL UNIQUE
);

CREATE TABLE IF NOT EXISTS authors (
    id      INTEGER PRIMARY KEY,
    name    TEXT NOT NULL UNIQUE
);

CREATE TABLE IF NOT EXISTS books (
    id          INTEGER PRIMARY KEY,
    title       TEXT NOT NULL,
    url         TEXT NOT NULL UNIQUE,
    price       REAL NOT NULL,
    rating      INTEGER NOT NULL CHECK (rating BETWEEN 1 AND 5),
    stock       INTEGER NOT NULL CHECK (stock IN (0,1)),
    description TEXT,
    category_id INTEGER NOT NULL,
    FOREIGN KEY (category_id) REFERENCES categories(id) ON DELETE RESTRICT
);

CREATE TABLE IF NOT EXISTS book_author (
    book_id   INTEGER NOT NULL,
    author_id INTEGER NOT NULL,
    PRIMARY KEY (book_id, author_id),
    FOREIGN KEY (book_id)   REFERENCES books(id)   ON DELETE CASCADE,
    FOREIGN KEY (author_id) REFERENCES authors(id) ON DELETE CASCADE
);
"""

# on delete restrict y on delete cascade

# 4. Ejecutamos todas las sentencias
conn.executescript(DDL)

print("✅ Tablas creadas correctamente")

conn.close()


✅ Tablas creadas correctamente


In [4]:
import sqlite3, json
from pathlib import Path

DB_PATH = "libros.db"              # cambia si tu DB se llama distinto
JSON_PATH = "libros_scrapeados.json"  # cambia si tu JSON se llama distinto

def cargar_categorias(db_path=DB_PATH, json_path=JSON_PATH):

    # 1. Cargar JSON
    data = json.loads(Path(JSON_PATH).read_text(encoding="utf-8"))
    print(f"Leídos {len(data)} libros desde el JSON ✅ \n")

    # 2. Conexión + FK on
    conn = sqlite3.connect(db_path)
    conn.execute("PRAGMA foreign_keys = ON;")
    cur = conn.cursor()

    # 3. recopilar categorías del JSON
    categorias = { item.get("category") for item in data if item.get("category") }
    print(f"categorías : {categorias}")

    # 4. Insertar en la tabla categories
    cur.executemany( "INSERT OR IGNORE INTO categories(name) VALUES (?);", [(c,) for c in sorted(categorias)])

    # 5. Guardar cambios
    conn.commit()

    # 6. Comprobar el resultado
    total = cur.execute("SELECT COUNT(*) FROM categories;").fetchone()[0]
    print(f"Listo categorías. Total en tabla: {total}")
    conn.close()



def cargar_authors(db_path=DB_PATH, json_path=JSON_PATH):
    # 1) Leer JSON -> Python
    data = json.loads(Path(json_path).read_text(encoding="utf-8"))

    # 2) Conexión y FK ON
    conn = sqlite3.connect(db_path)
    conn.execute("PRAGMA foreign_keys = ON;")
    cur = conn.cursor()

    # 3) Deduplicar autores (siempre lista)
    autores = set()
    for item in data:
        for nm in item["authors"]:
            autores.add(str(nm).strip())

    # 4) Insert masivo sin duplicar en DB
    cur.executemany(
        "INSERT OR IGNORE INTO authors(name) VALUES (?);",
        [(a,) for a in sorted(autores)]
    )

    # 5) Confirmar y mostrar total
    conn.commit()
    total = cur.execute("SELECT COUNT(*) FROM authors;").fetchone()[0]
    print(f"Listo autores. Total en tabla: {total}")
    conn.close()



def cargar_books(db_path=DB_PATH, json_path=JSON_PATH):
    data = json.loads(Path(json_path).read_text(encoding="utf-8"))
    conn = sqlite3.connect(db_path)
    conn.execute("PRAGMA foreign_keys = ON;")
    cur = conn.cursor()

    # 1) Obtener mapa categoría nombre->id
    # Precache de categorías name->id para acelerar
    cat_map = dict(cur.execute("SELECT name, id FROM categories;").fetchall())

    # 2) Sentencia de inserción
    insert_sql = """
    INSERT OR IGNORE INTO books(title, url, price, rating, stock, description, category_id)
    VALUES (?, ?, ?, ?, ?, ?, ?);
    """

    # 3) Preparar filas
    filas = []
    for it in data:
        cat = it.get("category")
        cat_id = cat_map.get(cat)

        # Agregar fila, en tupla para la tabla
        # Cada tupla representa una fila a insertar
        filas.append((
            it.get("title"),
            it.get("URL"),
            float(it.get("price") or 0.0),
            int(it.get("rating") or 0),
            int(it.get("stock") or 0),
            it.get("description"),
            cat_id
        ))
        # print(f"Preparando libro: {it.get('title')} (cat_id={cat_id})")

    # 4) Insertar todo
    cur.executemany(insert_sql, filas)
    conn.commit()



    # 5) Confirmar y mostrar total
    total = cur.execute("SELECT COUNT(*) FROM books;").fetchone()[0]
    print(f"Listo books. Total en tabla: {total}")
    conn.close()


def cargar_book_author(db_path=DB_PATH, json_path=JSON_PATH):
    data = json.loads(Path(json_path).read_text(encoding="utf-8"))

    conn = sqlite3.connect(db_path)
    conn.execute("PRAGMA foreign_keys = ON;")
    cur = conn.cursor()


    # 1) obtener mapas libro -> id y autor -> id
    # Cache: url libro -> id
    book_map = dict(cur.execute("SELECT url, id FROM books;").fetchall())
    # Cache: autor -> id
    author_map = dict(cur.execute("SELECT name, id FROM authors;").fetchall())


    # 2) sentencia inserción
    insert_rel = "INSERT OR IGNORE INTO book_author(book_id, author_id) VALUES (?, ?);"

    n_rel = 0
    # 3) recorrer datos del JSON 
    for it in data:
        # 4) obtener book_id por URL
        url = it.get("URL")
        book_id = book_map.get(url)

        # 5) obtener lista de autores
        autores = it.get("authors") or []

        # 6) iterar autores 
        for a in autores:
            # limpiar nombre
            a = str(a).strip()
            # obtener author_id
            a_id = author_map.get(a)

            # insertar relación
            cur.execute(insert_rel, (book_id, a_id))
            n_rel += 1

    # 7) confirmar y mostrar totales
    conn.commit()
    total_rel = cur.execute("SELECT COUNT(*) FROM book_author;").fetchone()[0]
    print(f"Listo relaciones. Agregadas ahora: {n_rel}. Total en tabla: {total_rel}")
    conn.close()



In [5]:
cargar_categorias()
cargar_authors()
cargar_books()
cargar_book_author()



Leídos 1000 libros desde el JSON ✅ 

categorías : {'Psychology', 'Historical', 'Short Stories', 'Autobiography', 'Health', 'Mystery', 'Sports and Games', 'Novels', 'Food and Drink', 'Sequential Art', 'Music', 'Art', 'History', 'Horror', 'Suspense', 'Adult Fiction', 'Poetry', 'Classics', 'Academic', 'Contemporary', 'Travel', 'Nonfiction', 'Humor', 'Romance', 'Fiction', 'Spirituality', 'Christian Fiction', 'Default', 'Fantasy', 'Self Help', 'Cultural', 'Add a comment', 'Young Adult', 'Science Fiction', 'Erotica', 'Science', 'Thriller', 'Politics', 'Philosophy', 'New Adult', 'Historical Fiction', 'Paranormal', 'Business', 'Parenting', 'Childrens', 'Christian', 'Crime', 'Womens Fiction', 'Religion', 'Biography'}
Listo categorías. Total en tabla: 50
Listo autores. Total en tabla: 867
Listo books. Total en tabla: 1000
Listo relaciones. Agregadas ahora: 1499. Total en tabla: 1499


# Consultas

In [7]:
import sqlite3, pandas as pd
def run(sql, params=()):
    conn = sqlite3.connect("libros.db")
    display(pd.read_sql_query(sql, conn, params=params))
    conn.close()


In [21]:
# 1. Libros con más de 3 estrellas y a menos de £12
%time
query = """
SELECT title, price, rating
FROM books
WHERE rating > 3 AND price < 12;
"""
run(query)

# SELECT sirve para seleccionar datos de una base de datos.
# FROM es para especificar la tabla de donde se obtienen los datos.
# WHERE sirve para filtrar los resultados según una condición.


CPU times: total: 0 ns
Wall time: 13.8 μs


,title,price,rating
0,I Am Pilgrim (Pilgrim #1),10.60,4
1,City of Fallen Angels (The Mortal Instruments #4),11.23,4
2,"The Sleep Revolution: Transforming Your Life, ...",11.68,4
3,"NaNo What Now? Finding your editing process, r...",10.41,4
4,History of Beauty,10.29,4
5,The Origin of Species,10.01,4
6,Green Eggs and Ham (Beginner Books B-16),10.79,4
7,Superman Vol. 1: Before Truth (Superman by Gen...,11.89,5
8,Old School (Diary of a Wimpy Kid #10),11.83,5
9,Greek Mythic History,10.23,5


In [22]:
# 2. Libros con precio > 50
%time
query = """
SELECT title, price
FROM books
WHERE price > 50
Limit 5;
"""
run(query)

# LIMIT sirve para limitar la cantidad de resultados devueltos por una consulta.


CPU times: total: 0 ns
Wall time: 15.7 μs


,title,price
0,Soumission,50.10
1,Rogue Lawyer (Rogue Lawyer #1),50.11
2,The Pilgrim's Progress,50.26
3,"We Love You, Charlie Freeman",50.27
4,Nightstruck: A Novel,50.35


In [23]:
# 3. Autores con más libros publicados (top 10)
%time
query = """
SELECT a.name AS author, COUNT(ba.book_id) AS num_books
FROM authors a
JOIN book_author ba ON a.id = ba.author_id
GROUP BY a.id
ORDER BY num_books DESC
LIMIT 10;
"""
run(query)

# JOIN se utiliza para combinar filas de dos o más tablas basándose en una columna relacionada entre ellas.
# GROUP BY se utiliza para agrupar filas que tienen los mismos valores en columnas especificadas.
# ORDER BY se utiliza para ordenar los resultados de una consulta en orden ascendente o descendente.
# AS se utiliza para asignar un alias a una columna o tabla en una consulta SQL.


CPU times: total: 0 ns
Wall time: 12.9 μs


,author,num_books
0,Desconocido,558
1,Stephen King,8
2,Irb Media,6
3,David Levithan,5
4,Bookhabits,4
5,David Lee,4
6,David Sedaris,4
7,Gillian Flynn,4
8,Sophie Kinsella,4
9,Betty Neels,3


In [24]:
# 4. Benchmark SIMPLE: Búsqueda de libros que empiezan con 'T'

import time
import sqlite3, pandas as pd

def run(sql, params=()):
    conn = sqlite3.connect("libros.db")
    display(pd.read_sql_query(sql, conn, params=params))
    conn.close()

print("=" * 70)
print("🐌 CONSULTA SIN ÍNDICE - Libros que empiezan con 'T'")
print("=" * 70)

# Consulta simple SIN índice
query_sin_indice = """
SELECT title, price, rating
FROM books
WHERE title LIKE 'T%'
ORDER BY title;
"""

conn = sqlite3.connect("libros.db")
cur = conn.cursor()

start = time.time()
run(query_sin_indice)
tiempo_sin_indice = time.time() - start
print(f"\n⏱️ Tiempo sin índice: {tiempo_sin_indice:.4f} segundos\n")

# ============================================================
# CREAR UN SOLO ÍNDICE EN 'title'
# ============================================================
print("=" * 70)
print("🔧 CREANDO ÍNDICE EN title")
print("=" * 70)

cur.execute("CREATE INDEX IF NOT EXISTS idx_books_title ON books(title);")

conn.commit()
conn.close()
print("✅ Índice creado\n")

# ============================================================
# MISMA CONSULTA CON ÍNDICE
# ============================================================
print("=" * 70)
print("🚀 CONSULTA CON ÍNDICE - Libros que empiezan con 'T'")
print("=" * 70)

query_con_indice = """
SELECT title, price, rating
FROM books
WHERE title LIKE 'T%'
ORDER BY title;
"""

start = time.time()
run(query_con_indice)
tiempo_con_indice = time.time() - start
print(f"\n⏱️ Tiempo con índice: {tiempo_con_indice:.4f} segundos\n")

# ============================================================
# COMPARACIÓN
# ============================================================
print("=" * 70)
print("📊 RESUMEN")
print("=" * 70)
print(f"Sin índice: {tiempo_sin_indice:.4f} seg")
print(f"Con índice: {tiempo_con_indice:.4f} seg")

if tiempo_sin_indice > tiempo_con_indice:
    mejora = ((tiempo_sin_indice - tiempo_con_indice) / tiempo_sin_indice) * 100
    print(f"✅ Mejora: {mejora:.1f}% más rápido")
else:
    print("ℹ️ Diferencia mínima en bases pequeñas")
print("=" * 70)

🐌 CONSULTA SIN ÍNDICE - Libros que empiezan con 'T'


,title,price,rating
0,Take Me Home Tonight (Rock Star Romance #3),53.98,3
1,Take Me with You,45.21,3
2,Taking Shots (Assassins #1),18.88,2
3,Talking to Girls About Duran Duran: One Young ...,25.15,4
4,Tastes Like Fear (DI Marnie Rome #3),10.69,1
...,...,...,...
302,Twenty Love Poems and a Song of Despair,30.95,4
303,Twenty Yawns,22.08,2
304,Twilight (Twilight #1),41.93,2
305,Two Boys Kissing,32.74,2



⏱️ Tiempo sin índice: 0.0410 segundos

🔧 CREANDO ÍNDICE EN title
✅ Índice creado

🚀 CONSULTA CON ÍNDICE - Libros que empiezan con 'T'


,title,price,rating
0,Take Me Home Tonight (Rock Star Romance #3),53.98,3
1,Take Me with You,45.21,3
2,Taking Shots (Assassins #1),18.88,2
3,Talking to Girls About Duran Duran: One Young ...,25.15,4
4,Tastes Like Fear (DI Marnie Rome #3),10.69,1
...,...,...,...
302,Twenty Love Poems and a Song of Despair,30.95,4
303,Twenty Yawns,22.08,2
304,Twilight (Twilight #1),41.93,2
305,Two Boys Kissing,32.74,2



⏱️ Tiempo con índice: 0.0492 segundos

📊 RESUMEN
Sin índice: 0.0410 seg
Con índice: 0.0492 seg
ℹ️ Diferencia mínima en bases pequeñas


In [25]:
# 5. categorias con más de 20 libros
%time
query = """
SELECT c.name, COUNT(*) AS total_libros
FROM categories c
JOIN books b ON c.id = b.category_id
GROUP BY c.id
HAVING total_libros > 20
ORDER BY total_libros DESC;
"""
run(query)

# HAVING se utiliza para filtrar grupos de resultados después de aplicar una función de agregación, como COUNT o SUM.


CPU times: total: 0 ns
Wall time: 16 μs


,name,total_libros
0,Default,152
1,Nonfiction,110
2,Sequential Art,75
3,Add a comment,67
4,Fiction,65
5,Young Adult,54
6,Fantasy,48
7,Romance,35
8,Mystery,32
9,Food and Drink,30
